# Credit Scoring (Thin Notebook)

This notebook simply *calls* the modular pipeline to keep code organized.

In [ ]:
# Adjust if running from a different working directory
%load_ext autoreload
%autoreload 2

from pathlib import Path
root = Path.cwd().parents[0]  # project root
root

In [ ]:
# Run the pipeline CLI-like steps from Python
from src.pipeline.steps import (
    Paths, ensure_dirs, load_raw, clean_df, detect_and_binarize_target,
    split_df, build_transformer, fit_transform, train_xgb, evaluate_model, shap_explain
)

P = Paths.from_root(root)
ensure_dirs(P)

df = load_raw(P)
df = clean_df(df)
df, target_col = detect_and_binarize_target(df)
train_df, test_df = split_df(df, target_col=target_col)

pre, *_ = build_transformer(train_df, target_col)
X_train, y_train, X_test, y_test = fit_transform(pre, train_df, test_df, target_col)

clf, spw = train_xgb(X_train, y_train)
clf.save_model(str(P.model_path))

metrics = evaluate_model(clf, X_test, y_test, P.reports_dir)
shap_explain(clf, pre, X_train, P.reports_dir)

metrics